In [1]:
import asyncio
import time
import aiohttp
from functools import wraps

In [2]:
def async_timeit(coro):
    @wraps(coro)
    async def inner(*args, **kwargs):
        t1 = time.time()
        try:
            return await coro(*args, **kwargs)
        finally:
            t2 = time.time()
            print(f"Time {coro.__name__}:", t2 - t1)

    return inner

In [6]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 50
URLS = [URL] * N
WORKERS = 5


async def fetch_url(url, session):
    async with session.get(url) as resp:
        return (resp.status, len(await resp.text()))


@async_timeit
async def worker_fetch_urls(que, session, name):
    print(f"started -- {name}")
    resps = []
    cnt = 0

    while True:
        url = await que.get()

        resps.append(await fetch_url(url, session))
        cnt += 1
        que.task_done()

    print(f"finished -- {name}, {cnt=}")
    return resps


@async_timeit
async def run():
    que = asyncio.Queue()
    for url in URLS:
        await que.put(url)

    async with aiohttp.ClientSession() as session:
        workers = [
            asyncio.create_task(
                worker_fetch_urls(que, session, f"fetcher_{i}")
            )
            for i in range(WORKERS)
        ]
        await que.join()

        for worker in workers:
            worker.cancel()

    print("total_resps")


await run()

started -- fetcher_0
started -- fetcher_1
started -- fetcher_2
started -- fetcher_3
started -- fetcher_4
Time worker_fetch_urls: 1.3515021800994873
Time worker_fetch_urls: 1.351201057434082
Time worker_fetch_urls: 1.3511500358581543
Time worker_fetch_urls: 1.351121187210083
Time worker_fetch_urls: 1.351097822189331
total_resps
Time run: 1.3535747528076172


In [8]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 50
URLS = [URL] * N
WORKERS = 10
BATCH_SIZE = N // WORKERS


async def fetch_url(url, session, sem):
    async with sem:
        async with session.get(url) as resp:
            return (resp.status, len(await resp.text()))


@async_timeit
async def fetch_batch_urls(urls, session, sem, name):
    print(f"started -- {name}")
    resps = []

    for url in urls:
        resps.append(await fetch_url(url, session, sem))

    print(f"finished -- {name}")
    return resps


@async_timeit
async def run():
    sem = asyncio.Semaphore(5)
    
    async with aiohttp.ClientSession() as session:
        coros = [
            fetch_batch_urls(
                URLS[i * BATCH_SIZE : (i + 1) * BATCH_SIZE],
                session,
                sem,
                f"fetcher_{i}",
            )
            for i in range(WORKERS)
        ]
        total = await asyncio.gather(*coros)

    print("total_resps", total)


await run()

started -- fetcher_0
started -- fetcher_1
started -- fetcher_2
started -- fetcher_3
started -- fetcher_4
started -- fetcher_5
started -- fetcher_6
started -- fetcher_7
started -- fetcher_8
started -- fetcher_9
finished -- fetcher_1
Time fetch_batch_urls: 1.045564889907837
finished -- fetcher_4
Time fetch_batch_urls: 1.0837488174438477
finished -- fetcher_2
Time fetch_batch_urls: 1.134119987487793
finished -- fetcher_3
Time fetch_batch_urls: 1.1588151454925537
finished -- fetcher_6
Time fetch_batch_urls: 1.2101759910583496
finished -- fetcher_0
Time fetch_batch_urls: 1.2123572826385498
finished -- fetcher_5
Time fetch_batch_urls: 1.2477240562438965
finished -- fetcher_8
Time fetch_batch_urls: 1.2563388347625732
finished -- fetcher_7
Time fetch_batch_urls: 1.275827169418335
finished -- fetcher_9
Time fetch_batch_urls: 1.2849581241607666
total_resps [[(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840)], [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (2

In [12]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 20
URLS = [URL] * N


@async_timeit
async def fetch_url(url, session, name):
    async with session.get(url) as resp:
        return (resp.status, len(await resp.text()))


@async_timeit
async def fetch_batch_urls(urls):
    print("started -- fetch_batch_urls")

    async with aiohttp.ClientSession() as session:
        coros = [
            fetch_url(url, session, f"fetcher_{i}")
            for i, url in enumerate(urls)
        ]
        total = await asyncio.gather(*coros)

    print("finished -- fetch_batch_urls")

    return total


@async_timeit
async def run():
    total = await fetch_batch_urls(URLS)
    print("total_resps", total)


await run()

started -- fetch_batch_urls
Time fetch_url: 1.0848100185394287
Time fetch_url: 1.2567939758300781
Time fetch_url: 1.2689580917358398
Time fetch_url: 1.2767841815948486
Time fetch_url: 1.2775070667266846
Time fetch_url: 1.2823560237884521
Time fetch_url: 1.2842509746551514
Time fetch_url: 1.284618854522705
Time fetch_url: 1.296581745147705
Time fetch_url: 1.3028841018676758
Time fetch_url: 1.3066086769104004
Time fetch_url: 1.3080840110778809
Time fetch_url: 1.3124120235443115
Time fetch_url: 1.313904047012329
Time fetch_url: 1.314032793045044
Time fetch_url: 1.3912949562072754
Time fetch_url: 1.408050298690796
Time fetch_url: 1.4355950355529785
Time fetch_url: 1.4381413459777832
Time fetch_url: 1.8171226978302002
finished -- fetch_batch_urls
Time fetch_batch_urls: 1.821415901184082
total_resps [(200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840), (200, 554840),

In [13]:
async def counter(n):
    for i in range(n):
        yield i
        print(f"yield {i=}")
        await asyncio.sleep(0.1)

In [14]:
async for x in counter(5):
    print(f"{x=}")

x=0
yield i=0
x=1
yield i=1
x=2
yield i=2
x=3
yield i=3
x=4
yield i=4


In [16]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 20
URLS = [URL] * N


@async_timeit
async def fetch_url(url, session, name):
    async with session.get(url) as resp:
        return (resp.status, len(await resp.text()), name)


async def fetch_batch_urls(urls):
    print("started -- fetch_batch_urls")

    async with aiohttp.ClientSession() as session:
        coros = [
            fetch_url(url, session, f"fetcher_{i}")
            for i, url in enumerate(urls)
        ]

        start = time.time()
        for task in asyncio.as_completed(coros):
            print(f"got task", time.time() - start)
            res = await task
            print(f"res after await", time.time() - start)

            name = res[2]
            num = int(name.split("_")[1])

            if num % 2 == 0:
                yield res

    print("finished -- fetch_batch_urls")


@async_timeit
async def run():
    total = []
    
    async for res in fetch_batch_urls(URLS):
        total.append(res)
    
    print("total_resps", total)


await run()

started -- fetch_batch_urls
got task 0.00017309188842773438
Time fetch_url: 0.6590638160705566
res after await 0.6615920066833496
got task 0.6616389751434326
Time fetch_url: 0.6666250228881836
res after await 0.6679520606994629
got task 0.6679911613464355
Time fetch_url: 1.0400850772857666
res after await 1.0421209335327148
got task 1.0421760082244873
Time fetch_url: 1.1730000972747803
res after await 1.1739819049835205
got task 1.1740212440490723
Time fetch_url: 1.1793041229248047
Time fetch_url: 1.1793463230133057
Time fetch_url: 1.180462121963501
Time fetch_url: 1.1814799308776855
Time fetch_url: 1.181689977645874
Time fetch_url: 1.182568073272705
res after await 1.1844842433929443
got task 1.184520959854126
res after await 1.1845371723175049
got task 1.1845510005950928
res after await 1.184568166732788
got task 1.184582233428955
res after await 1.1845991611480713
got task 1.184614896774292
res after await 1.184628963470459
got task 1.184645175933838
res after await 1.18465900421142

In [19]:
URL = "https://docs.python.org/3/whatsnew/3.14.html"
N = 10
URLS = [URL] * N


@async_timeit
async def fetch_url(url, session, name):
    async with session.get(url) as resp:
        return (resp.status, len(await resp.text()), name)


async def fetch_batch_urls(urls):
    print("started -- fetch_batch_urls")

    async with aiohttp.ClientSession() as session:
        coros = [
            fetch_url(url, session, f"fetcher_{i}")
            for i, url in enumerate(urls)
        ]

        start = time.time()
        async for task in asyncio.as_completed(coros):
            print(f"got task", time.time() - start)
            res = await task
            print(f"res after await", time.time() - start)

            name = res[2]
            num = int(name.split("_")[1])

            if num % 2 == 0:
                yield res

    print("finished -- fetch_batch_urls")


@async_timeit
async def run():
    total = []
    
    async for res in fetch_batch_urls(URLS):
        total.append(res)
    
    print("total_resps", total)


await run()

started -- fetch_batch_urls
Time fetch_url: 0.5032761096954346
Time fetch_url: 0.5045890808105469
got task 0.5056068897247314
res after await 0.5056328773498535
got task 0.5057117938995361
res after await 0.5057368278503418
Time fetch_url: 0.6091556549072266
got task 0.6101419925689697
res after await 0.6101670265197754
Time fetch_url: 0.62105393409729
got task 0.6222620010375977
res after await 0.6222751140594482
Time fetch_url: 0.6261839866638184
got task 0.6266059875488281
res after await 0.6266210079193115
Time fetch_url: 0.6761841773986816
got task 0.6770749092102051
res after await 0.6770999431610107
Time fetch_url: 0.6834900379180908
got task 0.6855790615081787
res after await 0.6855878829956055
Time fetch_url: 0.7134542465209961
got task 0.7150349617004395
res after await 0.7150709629058838
Time fetch_url: 0.7559680938720703
got task 0.7569148540496826
res after await 0.7569327354431152
Time fetch_url: 0.7780489921569824
got task 0.7792088985443115
res after await 0.77923297882

In [22]:
@async_timeit
async def nonblocking_api(delay, what):
    print(f"start nonblocking_api", what)
    await asyncio.sleep(delay)
    print(f"finish nonblocking_api", what)


@async_timeit
async def blocking_api(delay, what):
    print(f"start blocking_api", what)
    time.sleep(delay)
    print(f"finish blocking_api", what)


@async_timeit
async def main():
    task1 = asyncio.create_task(nonblocking_api(1, "hello"))
    task2 = asyncio.create_task(blocking_api(2, "world"))

    await asyncio.gather(task1, task2)


await main()

start nonblocking_api hello
start blocking_api world
finish blocking_api world
Time blocking_api: 2.0052757263183594
finish nonblocking_api hello
Time nonblocking_api: 2.006410598754883
Time main: 2.006782054901123


In [23]:
@async_timeit
async def nonblocking_api(delay, what):
    print(f"start nonblocking_api", what)
    await asyncio.sleep(delay)
    print(f"finish nonblocking_api", what)


@async_timeit
async def blocking_api(delay, what):
    print(f"start blocking_api", what)
    time.sleep(delay)
    # requests.get()
    # urllib.urlopen
    # factorial(100000)
    print(f"finish blocking_api", what)


@async_timeit
async def main():
    
    task1 = asyncio.create_task(blocking_api(2, "world"))
    task2 = asyncio.create_task(nonblocking_api(1, "hello"))

    await asyncio.gather(task1, task2)


await main()

start blocking_api world
finish blocking_api world
Time blocking_api: 2.005427122116089
start nonblocking_api hello
finish nonblocking_api hello
Time nonblocking_api: 1.001377820968628
Time main: 3.007410764694214


In [24]:
loop = asyncio.get_running_loop()

In [25]:
loop.run_in_executor?

Signature: loop.run_in_executor(executor, func, *args)
Docstring: <no docstring>
File:      /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/asyncio/base_events.py
Type:      method

In [27]:
@async_timeit
async def nonblocking_api(delay, what):
    print(f"start nonblocking_api", what)
    await asyncio.sleep(delay)
    print(f"finish nonblocking_api", what)


@async_timeit
async def blocking_api(delay, what):
    print(f"start blocking_api", what)
    
    loop = asyncio.get_running_loop()
    await loop.run_in_executor(None, time.sleep, delay)

    #time.sleep(delay)
    # requests.get()
    # urllib.urlopen
    # factorial(100000)

    print(f"finish blocking_api", what)


@async_timeit
async def main():
    
    task1 = asyncio.create_task(blocking_api(2, "world"))
    task2 = asyncio.create_task(nonblocking_api(1, "hello"))

    await asyncio.gather(task1, task2)


await main()

start blocking_api world
start nonblocking_api hello
finish nonblocking_api hello
Time nonblocking_api: 1.0015501976013184
finish blocking_api world
Time blocking_api: 2.0042219161987305
Time main: 2.0050251483917236


In [29]:
@async_timeit
async def nonblocking_api(delay, what):
    print(f"start nonblocking_api", what)
    await asyncio.sleep(delay)
    print(f"finish nonblocking_api", what)


@async_timeit
async def main():
    
    task = await asyncio.create_task(nonblocking_api(5, "world"))
    print("finish main()")


task = asyncio.create_task(main())

await asyncio.sleep(2)

task.cancel()

start nonblocking_api world


True

Time nonblocking_api: 2.0035531520843506
Time main: 2.0077531337738037


In [30]:
@async_timeit
async def nonblocking_api(delay, what):
    print(f"start nonblocking_api", what)
    await asyncio.sleep(delay)
    print(f"finish nonblocking_api", what)


@async_timeit
async def main():
    
    task = await asyncio.shield(nonblocking_api(5, "world"))
    print("finish main()")


task = asyncio.create_task(main())

await asyncio.sleep(2)

task.cancel()

start nonblocking_api world


True

Time main: 2.002429723739624
finish nonblocking_api world
Time nonblocking_api: 5.001015663146973


In [33]:
@async_timeit
async def fetch_api(delay, what):
    print(f"start fetch_api", what)
    await asyncio.sleep(delay / 2)

    if what == "url_2":
        raise ValueError("wrong connection")

    await asyncio.sleep(delay / 2)

    print(f"finish fetch_api", what)


@async_timeit
async def main():
    
    task1 = fetch_api(4, "url_1")
    task2 = fetch_api(4, "url_2")
    task3 = fetch_api(4, "url_3")

    await asyncio.gather(task1, task2, task3)


await main()

start fetch_api url_1
start fetch_api url_2
start fetch_api url_3
Time fetch_api: 2.0013058185577393
Time main: 2.0027310848236084


ValueError: wrong connection

finish fetch_api url_1
Time fetch_api: 4.002647161483765
finish fetch_api url_3
Time fetch_api: 4.002416133880615


In [46]:
@async_timeit
async def fetch_api(delay, what):
    print(f"start fetch_api", what)
    await asyncio.sleep(delay / 2)

    if what == "url_2":
        raise ValueError("wrong connection")

    await asyncio.sleep(delay / 2)

    print(f"finish fetch_api", what)
    return f"result:{what}"


@async_timeit
async def main():

    try:
        async with asyncio.TaskGroup() as task_group:
            task1 = task_group.create_task(fetch_api(4, "url_1"))
            #task2 = task_group.create_task(fetch_api(4, "url_2"))
            task3 = task_group.create_task(fetch_api(4, "url_3"))
    except Exception as err:
        print(f"error {err=}")

    for task in (task1, task3):
        print(f"{task.get_name()=}, {task.cancelled()=}, {task.result()=}")


await main()

start fetch_api url_1
start fetch_api url_3
finish fetch_api url_1
Time fetch_api: 4.002638816833496
finish fetch_api url_3
Time fetch_api: 4.002645969390869
task.get_name()='Task-1097', task.cancelled()=False, task.result()='result:url_1'
task.get_name()='Task-1098', task.cancelled()=False, task.result()='result:url_3'
Time main: 4.003329753875732


In [49]:
@async_timeit
async def fetch_api(delay, what):
    print(f"start fetch_api", what)
    await asyncio.sleep(delay / 2)

    if what == "url_2":
        raise ValueError("wrong connection")

    await asyncio.sleep(delay / 2)

    print(f"finish fetch_api", what)
    return f"result:{what}"


@async_timeit
async def main():

    try:
        async with asyncio.TaskGroup() as task_group:
            task1 = task_group.create_task(fetch_api(1, "url_1"))
            task2 = task_group.create_task(fetch_api(4, "url_2"))
            task3 = task_group.create_task(fetch_api(4, "url_3"))
    except Exception as err:
        print(f"error {err=}")

    for task in (task1, task2, task3):
        print(f"{task.get_name()=}, {task.cancelled()=}")
        #print(f"{task.get_name()=}, {task.cancelled()=}, {task.result()=}")


await main()

start fetch_api url_1
start fetch_api url_2
start fetch_api url_3
finish fetch_api url_1
Time fetch_api: 1.0025339126586914
Time fetch_api: 2.0011699199676514
Time fetch_api: 2.001767158508301
error err=ExceptionGroup('unhandled errors in a TaskGroup', [ValueError('wrong connection')])
task.get_name()='Task-1115', task.cancelled()=False
task.get_name()='Task-1116', task.cancelled()=False
task.get_name()='Task-1117', task.cancelled()=True
Time main: 2.0023632049560547
